In [1]:
import sys
import os
from phoenix.otel import register
from openinference.instrumentation.openai import OpenAIInstrumentor

# Add the repo root to path so we can import from shared/
sys.path.append(os.path.abspath(".."))
from shared.shopbot_config import SHOPBOT_SYSTEM_PROMPT
from openai import OpenAI

client = OpenAI()

e:\Unfold Data Science\LinkedIn\ai-observability-monitoring-and-debugging-llms-in-production-27494003\aiobserve\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import phoenix as px
session = px.launch_app()
print(session.url)

C:\Users\amanr\AppData\Local\Programs\Python\Python312\Lib\contextlib.py:144: SAWarning: Skipped unsupported reflection of expression-based index ix_cumulative_llm_token_count_total
  next(self.gen)
C:\Users\amanr\AppData\Local\Programs\Python\Python312\Lib\contextlib.py:144: SAWarning: Skipped unsupported reflection of expression-based index ix_latency
  next(self.gen)
C:\Users\amanr\AppData\Local\Programs\Python\Python312\Lib\contextlib.py:144: SAWarning: Skipped unsupported reflection of expression-based index ix_spans_session_id
  next(self.gen)
boto3 is installed but aioboto3 is not. To use AWS Bedrock models in Playground, install aioboto3: pip install aioboto3


🌍 To view the Phoenix app in your browser, visit http://localhost:6006/
📖 For more information on how to use Phoenix, check out https://arize.com/docs/phoenix
http://localhost:6006/


In [3]:
tracer_provider = register(
    project_name="shopbot",
    endpoint="http://localhost:6006/v1/traces",
)
OpenAIInstrumentor().instrument(tracer_provider=tracer_provider)

OpenTelemetry Tracing Details
|  Phoenix Project: shopbot
|  Span Processor: SimpleSpanProcessor
|  Collector Endpoint: http://localhost:6006/v1/traces
|  Transport: HTTP + protobuf
|  Transport Headers: {}
|  
|  Using a default SpanProcessor. `add_span_processor` will overwrite this default.
|  
|  WARNING: It is strongly advised to use a BatchSpanProcessor in production environments.
|  
|  `register` has set this TracerProvider as the global OpenTelemetry default.
|  To disable this behavior, call `register` with `set_global_tracer_provider=False`.



In [4]:
POLICY_SNIPPETS = {
    "shipping": "Free shipping is available on all orders over $75.",
    "loyalty": "The StyleHive Rewards loyalty program is open to all registered members.",
    "support hours": "Customer support is available Monday to Friday, 9am-6pm EST.",
    "30-day return": "StyleHive accepts returns within 30 days of delivery for unworn items with tags attached.",
}

In [5]:
def retrieve_policy(question: str) -> str:
    for keyword, snippet in POLICY_SNIPPETS.items():
        if keyword in question.lower():
            return snippet
    return "No specific policy found."

def generate_answer(question: str) -> str:
    context = retrieve_policy(question)
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": SHOPBOT_SYSTEM_PROMPT},
            {"role": "user", "content": f"Context: {context}\n\nQuestion: {question}"},
        ],
    )
    return response.choices[0].message.content

answer = generate_answer("What's your return policy for items bought during the Black Friday sale?")
print(answer)

Our return policy applies to all items purchased at StyleHive, including those bought during the Black Friday sale. You can return unworn items with tags attached within 30 days from the delivery date for a full refund. The refund will be processed to your original payment method within 5–7 business days. If you have any more questions or need assistance with your return, feel free to ask!
